In [1]:
# 多业务 QA：配置要加载的文件 (路径, 业务名)
import pandas as pd
import numpy as np
from pathlib import Path

# 方式一：手动列出 (path, 业务名)
# FILE_LIST = [
#     ('data/1_30/kc_byfutures.csv', 'kc_byfutures'),
#     ('data/1_30/kc_bnfutures.csv', 'kc_bnfutures'),
#     ('data/1_30/by_byfutures.csv', 'by_byfutures'),
#     ('data/1_30/bn_bnfutures.csv', 'bn_bnfutures'),
# ]

# 方式二（可选）：自动扫描 data/1_30/*.csv，用文件名作业务名
BASE = Path('data/2_2')
FILE_LIST = [(str(p), p.stem) for p in sorted(BASE.glob('*.csv'))]
# 若同时要 platform，可: FILE_LIST = [('data/data_platform-strategy-list_1769496701.csv', 'platform')] + FILE_LIST

print('待加载业务:', [name for _, name in FILE_LIST])

待加载业务: ['bn_bnfutures', 'by_byfutures', 'kc_bnfutures', 'kc_byfutures']


In [2]:
# 加载并合并：每个文件加列「业务」，concat 成 df
dfs = []
for path, name in FILE_LIST:
    try:
        d = pd.read_csv(path)
        d['业务'] = name
        dfs.append(d)
        print(f'已加载: {path} -> 业务={name}, 行数={len(d)}')
    except Exception as e:
        print(f'跳过 {path}: {e}')
df = pd.concat(dfs, ignore_index=True)
print(f'合并后总行数: {len(df)}, 业务: {df["业务"].unique().tolist()}')

已加载: data/2_2/bn_bnfutures.csv -> 业务=bn_bnfutures, 行数=412
已加载: data/2_2/by_byfutures.csv -> 业务=by_byfutures, 行数=81
已加载: data/2_2/kc_bnfutures.csv -> 业务=kc_bnfutures, 行数=374
已加载: data/2_2/kc_byfutures.csv -> 业务=kc_byfutures, 行数=86
合并后总行数: 953, 业务: ['bn_bnfutures', 'by_byfutures', 'kc_bnfutures', 'kc_byfutures']


In [3]:
import re

# 1. Extract Meta Info
def extract_meta(strategy_id):
    match = re.search(r'mode_(\d+)$', strategy_id)
    mode_idx = int(match.group(1)) if match else 0
    return mode_idx

df['mode_index'] = df['调度名'].apply(extract_meta)
df['symbol'] = df['币对(标准)']

# 2. Type Conversion
df['调度最后成交时间'] = pd.to_datetime(df['调度最后成交时间'], errors='coerce')
df['调度最后更新时间'] = pd.to_datetime(df['调度最后更新时间'], errors='coerce')
cols_to_float = ['成交收益', '资金费收益', '手续费', '滑点', '当前持仓', '最大持仓', '敞口', '市场成交占比', '成交量']
for col in cols_to_float:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 3. Sort by 业务, symbol, mode_index
df = df.sort_values(by=['业务', 'symbol', 'mode_index'])

# 4. Aggregation (same logic as QA.ipynb)
def aggregate_strategy(x):
    latest = x.iloc[-1]
    total_spread_pnl = x['成交收益'].sum()
    total_commission = x['手续费'].sum()
    total_slippage = x['滑点'].sum()
    total_funding = latest['资金费收益']
    total_pnl = total_spread_pnl + total_funding + total_slippage - total_commission
    max_mode = x['mode_index'].max()
    current_pos = latest['当前持仓']
    max_pos = x['最大持仓'].max()
    last_trade = latest['调度最后成交时间']
    last_update = latest['调度最后更新时间']
    exposure = latest['敞口']
    market_share = latest['市场成交占比']
    roi_base = abs(max_pos) if abs(max_pos) > 0 else np.nan
    roi = total_pnl / roi_base if roi_base else 0
    return pd.Series({
        'mode_count': max_mode,
        'total_spread_pnl': total_spread_pnl,
        'total_funding': total_funding,
        'total_commission': total_commission,
        'total_slippage': total_slippage,
        'total_pnl': total_pnl,
        'current_pos': current_pos,
        'max_pos': max_pos,
        'roi': roi,
        'last_trade': last_trade,
        'last_update': last_update,
        'exposure': exposure,
        'market_share': market_share,
        'raw_slippage_cost': abs(total_slippage),
    })

df_grouped = df.groupby(['业务', 'symbol']).apply(aggregate_strategy).reset_index()

# 5. Derived Metrics
denom = df_grouped['total_pnl'].replace(0, np.nan)
df_grouped['cost_ratio'] = (df_grouped['total_commission'] + df_grouped['raw_slippage_cost']) / denom

df_grouped['pos_ratio'] = (df_grouped['current_pos'].abs() / df_grouped['max_pos'].abs()).fillna(0)
def get_status(row):
    if row['pos_ratio'] > 0.8:
        return 'Building/Full'
    elif row['pos_ratio'] < 0.1 and abs(row['current_pos']) > 1e-6:
        return 'Closing'
    elif abs(row['current_pos']) <= 1e-6:
        return 'Closed'
    else:
        return 'Holding'
df_grouped['status'] = df_grouped.apply(get_status, axis=1)

def get_pnl_source(row):
    if row['total_pnl'] < 0:
        return 'Loss'
    elif row['total_funding'] > row['total_spread_pnl']:
        return 'Funding Driven'
    else:
        return 'Spread Driven'
df_grouped['pnl_source'] = df_grouped.apply(get_pnl_source, axis=1)

df_grouped = df_grouped.sort_values(by='total_pnl', ascending=False)
print('df_grouped 行数:', len(df_grouped), '业务:', df_grouped['业务'].unique().tolist())

df_grouped 行数: 399 业务: ['kc_bnfutures', 'kc_byfutures', 'by_byfutures', 'bn_bnfutures']


In [4]:
# 选择要查看的业务：填业务名则只画该业务；填 None 则画全部
selected_business = 'bn_bnfutures'  # 可改为 None 或 'platform', 'kc_bnfutures' 等
if selected_business is None:
    plot_df = df_grouped.copy()
    plot_title_suffix = ' (全部业务)'
else:
    plot_df = df_grouped[df_grouped['业务'] == selected_business].copy()
    plot_title_suffix = f' ({selected_business})'
print(f'当前绘图数据: {len(plot_df)} 行', plot_title_suffix)

当前绘图数据: 115 行  (bn_bnfutures)


In [5]:
import plotly.graph_objects as go

def pnl_stacked_bar(plot_df, title):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=plot_df['symbol'],
        y=plot_df['total_spread_pnl'],
        name='价差收益',
        marker_color='#2ecc71',
    ))
    fig.add_trace(go.Bar(
        x=plot_df['symbol'],
        y=plot_df['total_funding'],
        name='资金费收益',
        marker_color='#3498db',
    ))
    fig.update_layout(
        barmode='stack',
        title=title,
        xaxis_title='币种 (Symbol)',
        yaxis_title='金额 (USDT)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        height=500,
        margin=dict(b=120),
        xaxis_tickangle=-45,
    )
    return fig

# 前20赚钱
plot_winners = plot_df.sort_values('total_pnl', ascending=False).head(20)
pnl_stacked_bar(plot_winners, f'盈亏构成 · 前20赚钱 {plot_title_suffix}').show()

# 前20亏钱
plot_losers = plot_df.sort_values('total_pnl', ascending=True).head(20)
pnl_stacked_bar(plot_losers, f'盈亏构成 · 前20亏钱 {plot_title_suffix}').show()

# 敞口 (前25)
exposure_df = plot_df.reindex(plot_df['exposure'].abs().sort_values(ascending=False).index).head(25)
fig_exposure = go.Figure(go.Bar(
    x=exposure_df['symbol'],
    y=exposure_df['exposure'],
    marker_color=exposure_df['exposure'].apply(lambda v: '#2ecc71' if v >= 0 else '#e74c3c'),
))
fig_exposure.update_layout(
    title=f'各币种敞口 {plot_title_suffix}',
    xaxis_title='币种 (Symbol)',
    yaxis_title='敞口 (USDT)',
    height=500,
    margin=dict(b=120),
    xaxis_tickangle=-45,
)
fig_exposure.show()

# 资金效率散点图
eff = plot_df.copy()
eff['max_pos_abs'] = eff['max_pos'].abs()
eff = eff[eff['max_pos_abs'] > 0]
if len(eff) > 0:
    size_ref = eff['total_pnl'].abs().max() or 1
    eff = eff.copy()
    eff['bubble_size'] = (eff['total_pnl'].abs() / size_ref * 80).clip(5, 80)
    fig_eff = go.Figure(go.Scatter(
        x=eff['max_pos_abs'],
        y=eff['roi'],
        mode='markers+text',
        text=eff['symbol'].str.replace('-USDT-FTR', ''),
        textposition='top center',
        marker=dict(
            size=eff['bubble_size'],
            color=eff['total_pnl'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title='总收益(USDT)'),
            line=dict(width=0.5, color='gray'),
        ),
        hovertemplate='%{text}<br>最大持仓: %{x:,.0f} USDT<br>ROI: %{y:.4f}<extra></extra>',
    ))
    fig_eff.update_layout(
        title=f'资金效率散点图 {plot_title_suffix}',
        xaxis_title='最大持仓 |Max Position| (USDT)',
        yaxis_title='总收益率 (ROI)',
        height=550,
        showlegend=False,
    )
    fig_eff.show()

# 长时间未更新且仍有持仓
ref_time = pd.Timestamp('2026-01-27')
stale_threshold = ref_time - pd.Timedelta(hours=24)
stale_with_pos = plot_df[
    (plot_df['last_update'] < stale_threshold) &
    (plot_df['current_pos'].abs() > 10)
].sort_values('current_pos')
fig_stale = go.Figure(go.Bar(
    x=stale_with_pos['symbol'],
    y=stale_with_pos['current_pos'],
    marker_color=stale_with_pos['current_pos'].apply(lambda v: '#e74c3c' if v < 0 else '#2ecc71'),
))
fig_stale.update_layout(
    title=f'长时间未更新且仍有持仓 {plot_title_suffix}',
    xaxis_title='币种 (Symbol)',
    yaxis_title='当前持仓 (USDT)',
    height=450,
    margin=dict(b=120),
    xaxis_tickangle=-45,
)
fig_stale.show()

In [ ]:
# 循环所有业务，将每个业务的图保存为可分享的交互式 HTML（文件名与业务名一致）
from pathlib import Path
import plotly.graph_objects as go

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def pnl_stacked_bar(plot_df, title):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=plot_df['symbol'], y=plot_df['total_spread_pnl'], name='价差收益', marker_color='#2ecc71'))
    fig.add_trace(go.Bar(x=plot_df['symbol'], y=plot_df['total_funding'], name='资金费收益', marker_color='#3498db'))
    fig.update_layout(barmode='stack', title=title, xaxis_title='币种 (Symbol)', yaxis_title='金额 (USDT)',
                      legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
                      height=500, margin=dict(b=120), xaxis_tickangle=-45)
    return fig

ref_time = pd.Timestamp('2026-01-27')
stale_threshold = ref_time - pd.Timedelta(hours=24)

for name in df_grouped['业务'].unique():
    plot_df = df_grouped[df_grouped['业务'] == name].copy()
    suffix = f' ({name})'
    # 文件名与业务名一致，用业务名作前缀
    base = name

    # 1. 前20赚钱
    f1 = pnl_stacked_bar(plot_df.sort_values('total_pnl', ascending=False).head(20), f'盈亏构成 · 前20赚钱{suffix}')
    f1.write_html(OUTPUT_DIR / f'{base}_盈亏构成_前20赚钱.html')

    # 2. 前20亏钱
    f2 = pnl_stacked_bar(plot_df.sort_values('total_pnl', ascending=True).head(20), f'盈亏构成 · 前20亏钱{suffix}')
    f2.write_html(OUTPUT_DIR / f'{base}_盈亏构成_前20亏钱.html')

    # 3. 敞口
    exposure_df = plot_df.reindex(plot_df['exposure'].abs().sort_values(ascending=False).index).head(25)
    f3 = go.Figure(go.Bar(x=exposure_df['symbol'], y=exposure_df['exposure'],
                          marker_color=exposure_df['exposure'].apply(lambda v: '#2ecc71' if v >= 0 else '#e74c3c')))
    f3.update_layout(title=f'各币种敞口{suffix}', xaxis_title='币种 (Symbol)', yaxis_title='敞口 (USDT)',
                     height=500, margin=dict(b=120), xaxis_tickangle=-45)
    f3.write_html(OUTPUT_DIR / f'{base}_敞口.html')

    # 4. 资金效率散点图
    eff = plot_df.copy()
    eff['max_pos_abs'] = eff['max_pos'].abs()
    eff = eff[eff['max_pos_abs'] > 0]
    if len(eff) > 0:
        size_ref = eff['total_pnl'].abs().max() or 1
        eff = eff.copy()
        eff['bubble_size'] = (eff['total_pnl'].abs() / size_ref * 80).clip(5, 80)
        f4 = go.Figure(go.Scatter(x=eff['max_pos_abs'], y=eff['roi'], mode='markers+text',
            text=eff['symbol'].str.replace('-USDT-FTR', ''), textposition='top center',
            marker=dict(size=eff['bubble_size'], color=eff['total_pnl'], colorscale='RdYlGn', showscale=True,
                        colorbar=dict(title='总收益(USDT)'), line=dict(width=0.5, color='gray')),
            hovertemplate='%{text}<br>最大持仓: %{x:,.0f} USDT<br>ROI: %{y:.4f}<extra></extra>'))
        f4.update_layout(title=f'资金效率散点图{suffix}', xaxis_title='最大持仓 |Max Position| (USDT)',
                         yaxis_title='总收益率 (ROI)', height=550, showlegend=False)
        f4.write_html(OUTPUT_DIR / f'{base}_资金效率散点图.html')

    # 5. 长时间未更新且仍有持仓
    stale_with_pos = plot_df[(plot_df['last_update'] < stale_threshold) & (plot_df['current_pos'].abs() > 10)].sort_values('current_pos')
    f5 = go.Figure(go.Bar(x=stale_with_pos['symbol'], y=stale_with_pos['current_pos'],
                          marker_color=stale_with_pos['current_pos'].apply(lambda v: '#e74c3c' if v < 0 else '#2ecc71')))
    f5.update_layout(title=f'长时间未更新且仍有持仓{suffix}', xaxis_title='币种 (Symbol)', yaxis_title='当前持仓 (USDT)',
                     height=450, margin=dict(b=120), xaxis_tickangle=-45)
    f5.write_html(OUTPUT_DIR / f'{base}_长时间未更新且仍有持仓.html')

    print(f'已保存: {base} -> {OUTPUT_DIR}/ ({base}_*.html)')

print(f'全部完成，文件在目录: {OUTPUT_DIR.absolute()}')